# Validação Fase 0 — 5 Vídeos Longos do FullCycle

 Notebook para validar a extração de transcrições usando GPU do Colab.

**Configuração:** Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Instalar dependências
!pip install -q faster-whisper yt-dlp

import time
import json
import os
from pathlib import Path
from faster_whisper import WhisperModel, BatchedInferencePipeline

print("Dependências instaladas!")

In [ ]:
# 2. Carregar modelo na GPU
model = WhisperModel("base", device="cuda", compute_type="int8")
batched_model = BatchedInferencePipeline(model=model)
print("Modelo carregado na GPU (T4)!"

In [ ]:
# 3. Definir 5 vídeos longos do FullCycle
VIDEOS = [
    {"id": "rzTiupYyRZM", "title": "Git Worktree: O Recurso Que a IA Transformou em Essencial", "duration": 2356},
    {"id": "gi7lwLe6uv8", "title": "Copilot Trabalhando SOZINHO? Background Agents", "duration": 1570},
    {"id": "6iu-MKtxPPA", "title": "Lovable e Vibecoding O Risco Que Ninguém Te Conta", "duration": 2953},
    {"id": "ZQxRZ8tXQcM", "title": "Meu ambiente de desenvolvimento para 2026", "duration": 2849},
    {"id": "NH_NtrN2BEs", "title": "Claude Code: Guia COMPLETO", "duration": 3147},
]

print(f"{len(VIDEOS)} vídeos para processar")
print(f"Duração total estimada: {sum(v['duration'] for v in VIDEOS)/60:.0f} min")

In [ ]:
# 4. Função de transcrição
def transcribe_video(video_id: str, title: str) -> dict:
    """Transcreve um vídeo do YouTube e retorna o resultado."""
    url = f"https://www.youtube.com/watch?v={video_id}"
    output_dir = Path(f"output/{video_id}")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Download do áudio
    print(f"\n  Download: {title[:50]}...")
    t0 = time.time()
    !yt-dlp -x --audio-format mp3 -o "temp_audio.%(ext)s" {url} 2>/dev/null
    download_time = time.time() - t0
    
    # Encontrar arquivo de áudio
    audio_file = None
    for f in Path('.').glob('temp_audio.*'):
        if f.suffix in ['.mp3', '.webm', '.m4a']:
            audio_file = f
            break
    
    if not audio_file:
        print(f"  Erro: áudio não encontrado")
        return None
    
    # Transcrever com GPU (batched)
    print(f"  Transcrevendo com GPU (batch_size=16)...")
    t1 = time.time()
    segments, info = batched_model.transcribe(str(audio_file), batch_size=16, beam_size=5)
    segments_list = list(segments)
    transcribe_time = time.time() - t1
    
    # Salvar SRT
    def format_ts(s):
        h = int(s // 3600)
        m = int((s % 3600) // 60)
        sec = int(s % 60)
        ms = round((s - int(s)) * 1000)
        return f"{h:02d}:{m:02d}:{sec:02d},{ms:03d}"
    
    srt_lines = []
    for i, seg in enumerate(segments_list, 1):
        srt_lines.append(str(i))
        srt_lines.append(f"{format_ts(seg.start)} --> {format_ts(seg.end)}")
        srt_lines.append(seg.text.strip())
        srt_lines.append("")
    
    srt_path = output_dir / "transcript.srt"
    srt_path.write_text("\n".join(srt_lines), encoding="utf-8")
    
    # Salvar texto
    plain_text = "\n".join(seg.text.strip() for seg in segments_list)
    text_path = output_dir / "full_text.txt"
    text_path.write_text(plain_text, encoding="utf-8")
    
    # Salvar metadata
    metadata = {
        "video_id": video_id,
        "title": info.title if hasattr(info, 'title') else title,
        "language": info.language,
        "duration": info.duration,
        "word_count": len(plain_text.split()),
        "segments": len(segments_list),
        "download_time": download_time,
        "transcribe_time": transcribe_time,
        "total_time": download_time + transcribe_time,
    }
    
    meta_path = output_dir / "metadata.json"
    meta_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    
    # Limpar áudio temporário
    audio_file.unlink(missing_ok=True)
    
    # Resultado
    speedup = info.duration / transcribe_time if transcribe_time > 0 else 0
    print(f"  ✓ Concluído: {len(plain_text.split())} palavras em {transcribe_time:.1f}s ({speedup:.1f}x realtime)")
    
    return metadata

print("Função de transcrição definida!")

In [ ]:
# 5. Processar os 5 vídeos
results = []
total_start = time.time()

for i, video in enumerate(VIDEOS, 1):
    print(f"\n{'='*60}")
    print(f"VÍDEO {i}/{len(VIDEOS)}: {video['title']}")
    print(f"Duração estimada: {video['duration']/60:.1f} min")
    print(f"{'='*60}")
    
    result = transcribe_video(video['id'], video['title'])
    if result:
        results.append(result)

total_time = time.time() - total_start

In [ ]:
# 6. Relatório final
print("\n" + "="*70)
print("RELATÓRIO DE VALIDAÇÃO — FASE 0")
print("="*70)

print(f"\nVídeos processados: {len(results)}/{len(VIDEOS)}")
print(f"Tempo total: {total_time:.1f}s ({total_time/60:.1f} min)")

print(f"\n{'Vídeo':<45} {'Duração':>8} {'Palavras':>10} {'Speed':>8}")
print("-"*75)

total_duration = 0
total_words = 0

for r in results:
    title = r['title'][:42] + "..." if len(r['title']) > 45 else r['title']
    duration = r['duration']
    words = r['word_count']
    speed = duration / r['transcribe_time'] if r['transcribe_time'] > 0 else 0
    
    print(f"{title:<45} {duration/60:>7.1f}m {words:>10} {speed:>7.1f}x")
    
    total_duration += duration
    total_words += words

print("-"*75)
print(f"{'TOTAL':<45} {total_duration/60:>7.1f}m {total_words:>10}")

print(f"\nAceleração vs CPU (1.67x): ~{speed/1.67:.0f}x mais rápido com GPU")
print(f"\nArquivos salvos em: output/<video_id>/")

In [ ]:
# 7. Download dos resultados (opcional)
# Descomente para baixar os arquivos

# !zip -r fullcycle_transcriptions.zip output/
# from google.colab import files
# files.download("fullcycle_transcriptions.zip")